In [61]:
import pandas as pd

## Reading the csv file

In [62]:
df = pd.read_csv('final_data.csv')

## Dropping unwanted column

In [63]:
df.drop('OrderValue', axis=1, inplace=True)
df.drop('OrderID', axis=1, inplace=True)
df.drop('CustomerID', axis=1, inplace=True)
df.drop('ProductID', axis=1, inplace=True)

## Filling the missing values

In [64]:
df['Age'] = df['Age'].fillna(df['Age'].median()).round().astype('int32')
df['City'] = df['City'].fillna(df['City'].mode()[0])

In [65]:
numerical_cols = ['Quantity', 'Discount', 'Age', 'UnitPrice', 'Sales']
categorical_cols = ['PaymentMethod', 'Status', 'City', 'CustomerSegment', 'ProductName', 'Category']
date_cols = ['OrderDate', 'SignupDate']

## Fixing Incorrect Data types

In [66]:
for col in numerical_cols:
    df[col] = df[col].astype('int32')

for col in categorical_cols:
    df[col] = df[col].astype('category')

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [67]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49222 entries, 0 to 49221
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   OrderDate        49222 non-null  datetime64[us]
 1   Quantity         49222 non-null  int32         
 2   Discount         49222 non-null  int32         
 3   PaymentMethod    49222 non-null  category      
 4   Status           49222 non-null  category      
 5   Age              49222 non-null  int32         
 6   City             49222 non-null  category      
 7   SignupDate       49222 non-null  datetime64[us]
 8   CustomerSegment  49222 non-null  category      
 9   ProductName      49222 non-null  category      
 10  Category         49222 non-null  category      
 11  UnitPrice        49222 non-null  int32         
 12  Sales            49222 non-null  int32         
dtypes: category(6), datetime64[us](2), int32(5)
memory usage: 2.0 MB


## Task 1 --> Creating new features 

In [68]:
df['AgeGroup'] = pd.cut(df['Age'], bins=[17, 25, 35, 50, 65], labels=['Young', 'Young Adult', 'Adult', 'Senior'])
df['GrossAmount'] = df['Quantity'] * df['UnitPrice']

In [69]:
df.head()

,OrderDate,Quantity,Discount,PaymentMethod,Status,Age,City,SignupDate,CustomerSegment,ProductName,Category,UnitPrice,Sales,AgeGroup,GrossAmount
0,2025-08-28,4,10,Gateway,Completed,36,Qom,2024-03-05,Regular,USB-C Cable,Accessories,9,32,Adult,36
1,2024-05-31,1,10,Wallet,Completed,49,Kerman,2025-06-04,Regular,Backpack,Accessories,42,38,Adult,42
2,2025-08-10,1,20,CardToCard,Completed,36,Tehran,2025-10-19,Regular,Mechanical Keyboard,Electronics,62,50,Adult,62
3,2024-10-11,1,10,Gateway,Completed,45,Shiraz,2023-12-07,Regular,Office Chair,Home Office,180,162,Adult,180
4,2024-02-19,2,0,Gateway,Completed,24,Karaj,2023-12-02,Regular,Phone Case,Accessories,14,28,Young,28


## Task 2 --> Extracting year, month and day -> Extracting length of str columns -> 

In [70]:
df['OrderYear'] = df['OrderDate'].dt.year
df['OrderMonth'] = df['OrderDate'].dt.month
df['OrderDay'] = df['OrderDate'].dt.day

In [71]:
df['ProductNameWords'] = df['ProductName'].str.split().str.len()

In [72]:
df.head()

,OrderDate,Quantity,Discount,PaymentMethod,Status,Age,City,SignupDate,CustomerSegment,ProductName,Category,UnitPrice,Sales,AgeGroup,GrossAmount,OrderYear,OrderMonth,OrderDay,ProductNameWords
0,2025-08-28,4,10,Gateway,Completed,36,Qom,2024-03-05,Regular,USB-C Cable,Accessories,9,32,Adult,36,2025,8,28,18
1,2024-05-31,1,10,Wallet,Completed,49,Kerman,2025-06-04,Regular,Backpack,Accessories,42,38,Adult,42,2024,5,31,12
2,2025-08-10,1,20,CardToCard,Completed,36,Tehran,2025-10-19,Regular,Mechanical Keyboard,Electronics,62,50,Adult,62,2025,8,10,26
3,2024-10-11,1,10,Gateway,Completed,45,Shiraz,2023-12-07,Regular,Office Chair,Home Office,180,162,Adult,180,2024,10,11,19
4,2024-02-19,2,0,Gateway,Completed,24,Karaj,2023-12-02,Regular,Phone Case,Accessories,14,28,Young,28,2024,2,19,17


## Task 3 --> One-Hot Encoding

In [ ]:
categorical_cols = ['PaymentMethod', 'Status', 'City', 'CustomerSegment', 'ProductName', 'Category', 'AgeGroup']

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

df_encoded

## Task 4 --> ColumnTransformer

In [84]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_features = ['PaymentMethod', 'Status', 'City', 'CustomerSegment', 'ProductName', 'Category', 'AgeGroup']
numerical_features = ['Quantity','Discount','Age','UnitPrice','GrossAmount','OrderYear','OrderMonth','OrderDay','ProductNameWords']

transformer = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', 'passthrough', numerical_features)
    ]
)

X = df[categorical_features + numerical_features]
X_transformed = transformer.fit_transform(X)

X_transformed.shape

(49222, 59)

## Task 5 --> StandardScaler

#### Standard Scaler transform numerical value so that they are approximately mean(0) and Standard deviation(1).
#### Formula => z = (x - mean) / standard deviation

In [87]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[numerical_features])
X_scaled.shape

(49222, 9)

## Task 6 --> MinMaxScaler

Xscaled​ = (X - Xmin) / Xmax​ − Xmin​​

In [91]:
from sklearn.preprocessing import MinMaxScaler

minmax = MinMaxScaler()
x_minmax = minmax.fit_transform(df[numerical_features])
x_minmax.shape

(49222, 9)

## Task 7 --> Preprocessing Pipeline

In [92]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

categorical_features = ['PaymentMethod', 'Status', 'City', 'CustomerSegment', 'ProductName', 'Category', 'AgeGroup']
numerical_features = ['Quantity','Discount','Age','UnitPrice','GrossAmount','OrderYear','OrderMonth','OrderDay','ProductNameWords']

numeric_pipeline = Pipeline(
    steps=[
        ('impute', SimpleImputer(strategy='mean')),
        ('scale', StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('encoding', OneHotEncoder(handle_unknown='ignore'))
    ]
)

features = ColumnTransformer(
    transformers=[
        ('Numerical Feature', numeric_pipeline, numerical_features),
        ('Categorical Features', categorical_pipeline, categorical_features)
    ]
)

features

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('Numerical Feature', ...), ('Categorical Features', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transfor

## Task 8 --> Full Scikit-learn Pipeline

In [94]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

numeric_pipeline = Pipeline(
    steps=[
        ('impute', SimpleImputer(strategy='mean')),
        ('scale', StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('encoding', OneHotEncoder(handle_unknown='ignore'))
    ]
)

features = ColumnTransformer(
    transformers=[
        ('Numerical Feature', numeric_pipeline, numerical_features),
        ('Categorical Features', categorical_pipeline, categorical_features)
    ]
)

model = Pipeline(
    steps=[
        ('features', features),
        ('regression', RandomForestRegressor())
    ]
)

X = df[numerical_features + categorical_features]
Y = df['Sales']

x_train, x_test, y_train, y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

model.fit(x_train, y_train)

y_predict = model.predict(x_test)

mean_absolute_error(y_test, y_predict)

0.03647536820721176

## Task 9 --> Pipeline Benefits

#### 1. Benefits of Pipeline
#### Makes the ML process simple and organized.
#### Automates preprocessing steps.
#### Reduces manual work and errors.
#### Makes the model easier to use and deploy.

#### 2. Problems 
#### Pipeline can be difficult to understand at first.
#### Debugging can be harder when many steps are combined.
#### A small mistake in one step can affect the whole pipeline.
#### It may take some time to set up properly.

#### Difference — Without Pipeline vs With Pipeline
#### Steps are done manually  | Steps are automated
#### More code  |  Less and organized code
#### Higher chance of errors  |  Fewer errors
#### Harder to maintain  |  Easier to maintain